In [ ]:
"""
Task 14: Compile and benchmark the custom SwiGLU CUDA kernel.
Requires an NVIDIA GPU + CUDA Toolkit installed (cannot run on CPU-only
sandboxes). Compiles swiglu_kernel.cu on the fly with PyTorch's JIT loader.
"""

import time
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load

swiglu_cuda = load(
    name="swiglu_cuda",
    sources=["task14_swiglu_kernel.cu"],
    verbose=True,
)

def swiglu_pytorch(x, gate):
    return F.silu(x) * gate

def benchmark(fn, x, gate, iters=200):
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(iters):
        fn(x, gate)
    torch.cuda.synchronize()
    return (time.time() - start) / iters

if __name__ == "__main__":
    device = "cuda"
    x = torch.randn(4096, 4096, device=device)
    gate = torch.randn(4096, 4096, device=device)

    custom_out = swiglu_cuda.forward(x, gate)
    ref_out = swiglu_pytorch(x, gate)
    print("Max abs diff vs PyTorch reference:", (custom_out - ref_out).abs().max().item())

    custom_time = benchmark(lambda a, b: swiglu_cuda.forward(a, b), x, gate)
    pytorch_time = benchmark(swiglu_pytorch, x, gate)

    print(f"Custom CUDA kernel: {custom_time*1000:.4f} ms/iter")
    print(f"Standard PyTorch:   {pytorch_time*1000:.4f} ms/iter")
    print(f"Speedup: {pytorch_time / custom_time:.2f}x")

**`task14_benchmark.py`**

**CUDA kernel source (`task14_swiglu_kernel.cu`)** — compiled via the code cell below:

```cpp
// swiglu_kernel.cu
// Vectorized SwiGLU activation computed directly on the GPU thread grid.
// SwiGLU(x, gate) = (x * sigmoid(x)) * gate      [ i.e. SiLU(x) * gate ]

#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void swiglu_kernel(const float* x, const float* gate, float* out, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        float xv = x[idx];
        float silu = xv / (1.0f + expf(-xv));   // SiLU(x) = x * sigmoid(x)
        out[idx] = silu * gate[idx];
    }
}

torch::Tensor swiglu_forward(torch::Tensor x, torch::Tensor gate) {
    auto out = torch::empty_like(x);
    int n = x.numel();

    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;

    swiglu_kernel<<<blocks, threads>>>(
        x.data_ptr<float>(),
        gate.data_ptr<float>(),
        out.data_ptr<float>(),
        n
    );

    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &swiglu_forward, "SwiGLU forward (CUDA)");
}

```

---
## Task 14: Custom CUDA Kernel for Accelerated SwiGLU Activation